In [ ]:
# 处理JP在途数据
import pandas as pd

In [ ]:
time = '20261116'

In [ ]:
jp_in_trans_total = pd.read_excel(rf'D:\shipping_recommendations2.0\src_data\JP在途-本地已出库\调拨单导出{time}-JP在途.xlsx',sheet_name='单据数据',usecols=['调拨单号','实际出库日期'])
jp_in_trans_detail = pd.read_excel(rf'D:\shipping_recommendations2.0\src_data\JP在途-本地已出库\调拨单导出{time}-JP在途.xlsx',sheet_name='明细数据')

In [ ]:
jp_in_trans_detail.head(5)

In [ ]:
merge_df = pd.merge(jp_in_trans_detail, jp_in_trans_total, how='left', left_on='调拨单号', right_on='调拨单号')

In [ ]:
merge_df.columns

In [ ]:
merge_df = merge_df[['平台站点', 'SKU', 'MSKU','调出数量', '调拨单号','物流方式', '实际出库日期']]
merge_df['状态'] = '已出运'
merge_df['ShipmentId'] = 'JP头程出库无ShipmentId'


In [ ]:
# 新增‘预计到仓日期’列，如果物流方式为‘空运’，则预计到仓日期为实际出库日期+15天，如果物流方式为‘海运’，则预计到仓日期为实际出库日期+25天
# 先将 ‘实际出库日期’列转换为日期类型
merge_df['实际出库日期'] = pd.to_datetime(merge_df['实际出库日期'])
merge_df['预计到仓日期'] = merge_df.apply(lambda x: x['实际出库日期'] + pd.Timedelta(days=15) if x['物流方式'] == '空运' else x['实际出库日期'] + pd.Timedelta(days=25), axis=1)

In [ ]:
# 新增仓库列，去掉‘AMAZON:’字符串，并在字符串尾部添加‘_FBA’字符串
merge_df['仓库'] = merge_df['平台站点'].str.replace('AMAZON:', '') + '_FBA'
# 重命名‘调出数量’，‘实际出库日期’，'调拨单号'列名
merge_df.rename(columns={'调出数量': '发货量', '实际出库日期': '预计出运日期', '调拨单号': '发货单号'}, inplace=True)

In [ ]:
merge_df['发货单号'] = '调拨单号：' + merge_df['发货单号'].astype(str)

In [ ]:
# 删除不需要的列
merge_df.drop(columns=['平台站点'], inplace=True)
merge_df.columns

In [ ]:
result_df = merge_df[['仓库', 'SKU', 'MSKU', '发货量', '预计到仓日期', '预计出运日期', '状态', '物流方式', 'ShipmentId', '发货单号']]
# 将 预计到仓日期 和 预计出运日期 列转换为字符串类型，格式为YYYY-MM-DD
result_df['预计到仓日期'] = result_df['预计到仓日期'].dt.strftime('%Y-%m-%d')
# result_df['预计出运日期'] = result_df['预计出运日期'].dt.strftime('%Y-%m-%d')

In [ ]:
result_df.to_excel(rf'D:\shipping_recommendations2.0\src_data\在途货件\JP在途-预计到仓结果{time}.xlsx', index=False)

In [ ]:
# 后续需考虑 JP站的 出现 在途货件物流方式为‘FBA自提物流’情况，是否与头程出库调拨单有重复性
# 还需考虑 若‘预计到仓日期’小于当前日期的情况，是否将到仓日期延后。-需根据追踪表灵活调整
# 确认JP的物流追踪信息更新时间点-已确认（）

In [ ]:
# 读取 在途货件表
# in_transit_shipments = pd.read_excel(f'D:/shipping_recommendations2.0/src_data/在途货件/在途货件{time}.xlsx', sheet_name='在途货件-正常')
in_transit_shipments = pd.read_excel(f'E:/sontu/shipping_recommendations/src_data/在途货件/在途货件{time}-新版.xlsx', sheet_name='在途货件-正常')

In [ ]:
# 将merge_df内容放入在途货件表中，这两个表列名完全一致
df = pd.concat([in_transit_shipments, result_df], ignore_index=True)
# 将 在途货件{DEFAULT_DATE}.xlsx的子表'在途货件-正常'内容替换为 df
with pd.ExcelWriter(f'E:/sontu/shipping_recommendations/src_data/在途货件/在途货件{time}-新版.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='在途货件-正常', index=False)

In [ ]:
# 读取 JP物流追踪信息表并读取子表：7月、8月
# jp_shipping_tracking_7 = pd.read_excel(r'D:\shipping_recommendations2.0\src_data\JP在途-本地已出库\2026发货跟踪表JP.xlsx', sheet_name='7月')
# jp_shipping_tracking_8 = pd.read_excel(r'D:\shipping_recommendations2.0\src_data\JP在途-本地已出库\2026发货跟踪表JP.xlsx', sheet_name='8月')

In [ ]:
# 将两个表合并
# jp_shipping_tracking = pd.concat([jp_shipping_tracking_7, jp_shipping_tracking_8], ignore_index=True)

In [ ]:
# jp_shipping_tracking.columns

In [ ]:
# jp_shipping_tracking = jp_shipping_tracking[['本地调拨单号', '送达海外仓时间', '海外仓发走时间']]
